In [ ]:
!pip install --upgrade pip torch torchvision torchaudio transformers diffusers accelerate

In [11]:
import os, time, gc
from datetime import datetime
import torch
from diffusers import StableDiffusion3Pipeline

print("--- PLATFORM CHECK ---")
print(f"Torch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print("--- GPU CHECK ---")
print(f"CUDA available?: {torch.cuda.is_available()}")
print(f"Device Name:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}")
print(f"Device Count:    {torch.cuda.device_count()}")
print(f"VRAM available:  {torch.cuda.mem_get_info(0)[0] / 1e9:.2f} GB")
print(f"VRAM used:       {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

--- PLATFORM CHECK ---
Torch version: 2.12.0+cu130
CUDA version: 13.0
--- GPU CHECK ---
CUDA available?: True
Device Name:     NVIDIA GeForce RTX 3090
Device Count:    1
VRAM available:  1.78 GB
VRAM used:       18.34 GB


In [5]:
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
# os.environ["HF_HOME"] = "/tmp/hf_home"
# os.environ["HF_CACHE"] = "/tmp/hf_cache"
!hf auth login --token $HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `runpod_torch` has been saved to /workspace/.cache/huggingface/stored_tokens
Your token has been saved to /workspace/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
class SD3Pipeline:
    def __init__(self, model="stabilityai/stable-diffusion-3.5-medium", low_vram=False):
        self.clear_vram()
        self.pipe = StableDiffusion3Pipeline.from_pretrained(
            model,
            torch_dtype=torch.float16
        )
        
        if low_vram:
            self.pipe.enable_model_cpu_offload()
        else:
            self.pipe = self.pipe.to("cuda")

    def generate(self, prompt: str, width=1280, height=768, steps=35, guidance=4.0):
        t1 = time.time()
        result = self.pipe(prompt=prompt, width=width, height=height,
                           num_inference_steps=steps, guidance_scale=guidance)
        ts = datetime.now().isoformat().split('.')[0]
        filename = f"{'_'.join(prompt.split()[:5])}_{ts}.png".replace(',', '').replace(':', '_')
        os.makedirs("output", exist_ok=True)
        result.images[0].save(f"output/{filename}")
        print(f"Generated {filename} in {time.time() - t1:.2f}s")
        return result.images[0]

    def unload(self):
        del self.pipe
        self.pipe = None
        self.clear_vram()

    @staticmethod
    def clear_vram():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [6]:
try:
    pipe.unload()
except:
    pass
pipe = SD3Pipeline()

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [9]:
prompts = [
    "A wide-angle cinematic shot looking down on a massive multi-level highway interchange construction site. Exposed rebar skeletons, giant concrete formwork, multiple massive yellow lattice-boom cranes, and hundreds of workers wearing high-visibility vests. Steam rising from fresh asphalt, dusty atmosphere, intense late afternoon sunlight creating strong geometric shadows.",
    "A low-angle perspective looking up at massive, freshly cast concrete box girders being guided into place over an existing busy highway by an enormous gantry crane. Temporary steel scaffolding supports the structure, bright orange construction vehicles, sparks flying from welding torches, overcast but bright daytime sky.",
    "A dramatic sunrise shot of a cable-stayed bridge construction site bridging a wide river. The main concrete towers are half-built, with temporary red steel supports. Massive spools of thick steel cable, tugboats moving construction barges below, heavy mist over the water, dramatic orange and purple sky, golden reflections.",
    "A claustrophobic, dramatic underwater view from within a flooded, pressurized tunnel construction zone. A colossal Tunnel Boring Machine (TBM) head with massive cutting teeth. Powerful underwater floodlights casting long beams through the murky, silt-filled water. Divers in heavy commercial gear working near hydraulic machinery, bubble plumes rising, industrial chaos submerged.",
    "A close-up view of a modern automobile assembly line, focused on industrial robotic arms. Bright blue sparks fly from robotic welding heads joining a car body panel. Shiny painted car chassis moving along a track, cool overhead LED lighting reflecting off metallic surfaces, background blur showing workers in uniforms, efficient and sterile environment.",
    "An extreme macro shot capturing the rapid motion of a precision pick-and-place machine's head placing a tiny surface-mount capacitor onto a green PCB. Vacuum nozzles, tiny spools of components, sophisticated computer vision cameras, focused white ring light highlighting the reflective solder, blurred background.",
    "A professional, clean product photograph on a matte, slate-grey surface. An organized electronics component kit with an Arduino Uno microcontroller, various colored LEDs, resistors with clear color bands, capacitors, jumper wires, and breadboards laid out geometrically. Perfect softbox lighting, shallow depth of field, high-resolution texture details.",
    "A cinematic close-up of a massive, retro-futuristic control console inside a hydroelectric dam's generation hall. Analog gauges with green backlighting, vintage heavy-duty toggles, illuminated push buttons, a detailed flow diagram etched into the panel. A view through a large window of turbulent white water gushing from turbines, soft, atmospheric lighting.",
    "A wide-angle landscape shot of a massive electrical transformer station at twilight. Colossal black oil-filled transformers, intricate networks of ceramic insulators, high-voltage copper busbars, steel lattice towers against a deep blue dusk sky, thousands of small lights illuminating the complex industrial matrix.",
    "A medium shot looking across a tidy power relay station during a thunderstorm. Hundreds of precise, synchronized switches, circuit breakers, and relays mounted on gray panels within weather-sealed cabinets. Rain slicked metallic surfaces, distant lightning flash illuminating the scene, ominous grey sky.",
    "A symmetrical shot looking down a long, cold data center server aisle. Rows of tall black server racks with thousands of tiny, blinking blue and green activity LEDs. Polished raised flooring, neat bundles of network cables, overhead cable trays. Cool, ambient lighting, sterile and high-tech atmosphere.",
    "A dynamic photo from the crowd of a massive outdoor live music festival at night. A huge stage with an explosive display of lasers, moving spotlights (pink, blue, yellow), and smoke machines. A famous band playing on stage, thousands of people with arms raised holding smartphones, energy, sweat, vibrant atmosphere.",
    "A wide shot of an exclusive art gallery opening event. Large, minimalist abstract expressionist paintings with rich textures hang on stark white walls, illuminated by precise track lighting. A sophisticated crowd holding wine glasses mingles, polished concrete floor, modern architecture, soft, inviting light.",
    "A picturesque medium shot looking across a vibrant mid-sized city park during the spring. Mature oak trees with lush green leaves, a winding gravel path, a ornate Victorian gazebo, diverse people picnicking, a small pond with ducks, low city skyline on the horizon under a soft blue sky.",
    "A vertical, high-resolution orthographic satellite image of a vibrant mid-sized city park. Clear separation between lush green tree canopies, geometric flower beds, winding brown paths, a blue artificial pond, a baseball diamond, surrounded by suburban grids and asphalt roads. Sunlight perfectly overhead, no distortion.",
    "A vast, high-resolution satellite image capturing a major metropolitan area at night. A dense spiderweb of golden and white light defining major highways, grid-like patterns of city streets, dark coastal outlines, the luminous glow of the downtown core, surrounded by the subtle transition to the darkness of rural lands.",
    "A cinematic landscape shot from a jagged, snow-capped mountaintop looking out over an endless, arid desert plain. The rocky mountain foreground is dark and textured, transitioning to rolling golden sand dunes and rugged mesa formations that stretch to the hazy horizon, deep blue sky.",
    "A dynamic, low-level photograph capturing a severe tropical storm making landfall on a coastal town. Palm trees are horizontal, bent violently by powerful wind and driving rain. Massive, grey-black supercell cloud formations, churning ocean with massive, chaotic brown waves, water flowing down streets, dark and ominous.",
    "A rich, lush photo capturing the interior of a temperate rainforest on a foggy day. Every surface is covered in vibrant green moss, ferns, and lichens. Massive old-growth Douglas fir and cedar trees, beams of soft, filtered sunlight cutting through the canopy fog, glistening wet textures, atmospheric perspective.",
    "A devastating aerial shot showing a powerful Category 5 hurricane’s tight, perfectly formed eye from high altitude. The massive, swirling, stacked cloud walls rotate violently around the calm blue eye, casting shadows. Dark, chaotic outer bands stretch to the horizon, cinematic and powerful.",
    "A dramatic photograph capturing a violent, charcoal-grey tornado funnel cloud twisting fiercely from a colossal supercell thunderstorm over flat Midwestern farmland. The vortex is dark, sucking up debris from the ground. Ominous, green-tinged storm clouds, lightning flashes, dust swirling at the base, terrifying scale.",
    "A minimalist, surreal photograph capturing the mirror-like surface of the Salar de Uyuni salt flats in Bolivia during the wet season. A thin layer of still water perfectly reflects the sky and the distant Andes mountains. A solitary figure walks across the seemingly infinite reflection, bright daylight, perfect symmetry, pure white and blue.",
    "A breathtaking photo of a volcanic eruption at night, generating its own electrical storm. A plume of ash and fire explodes violently into the dark sky. Dozens of powerful blue and purple lightning bolts strike within the ash cloud, illuminating the chaotic explosion. Red-hot lava flows down the mountain, terrifyingly beautiful.",
    "An aerial photograph of a brand new volcanic island emerging from the deep blue ocean. The central cone is actively smoking, glowing red-orange lava flows down into the sea creating steam plumes. Fresh, jagged black volcanic rock forms the coastline, isolated, raw geological creation.",
    "A stunning, high-angle aerial view of a perfect circular coral atoll. A vibrant turquoise lagoon is protected by the outer reef ring, contrasting sharply with the deep indigo ocean surrounding it. White sand beaches, lush green palm trees, a few small thatched-roof huts, clear tropical sunlight.",
    "A terrifying, low-angle shot from a coastal viewpoint, capturing a colossal tsunami wave towering hundreds of feet over a shoreline city. The massive, chaotic wall of water is grey and churning with debris. Small figures run frantically on the beach below, dark overcast sky, catastrophic scale, dynamic motion.",
    "A dramatic close-up photograph capturing chaotic, stormy ocean chop in the middle of a gale. Massive, grey-green waves collide, creating explosive white sea spray and deep troughs. Ominous, dark storm clouds, dynamic water textures, constant motion, spray obscuring the horizon.",
    "A breathtaking wide landscape shot inside a deep Norwegian fjord. Towering, snow-dusted mountains frame the scene. A massive humpback whale breaches completely out of the glassy, dark water, creating a giant splash. Multiple Orcas swim nearby, dramatic soft sunlight, tranquility interrupted by raw power.",
    "A panoramic shot from the deck of a small, red expedition boat moving silently through a narrow Norwegian fjord. Massive, vertical cliffs covered in waterfalls tower on both sides. Clear, still water perfectly reflects the snow-capped peaks. Cool, soft light, deep blues and greens, peaceful solitude.",
    "An otherworldly underwater photo from beneath a massive Antarctic ice shelf. The ice overhead is a brilliant, deep cobalt blue with complex trapped air bubbles. A technical diver in a drysuit and multiple tanks explores the dark abyss below, powerful dive lights cut through the crystal-clear water, awe-inspiring scale.",
    "A dark, pressurized photograph from a deep-sea submersible inside a deep ocean trench. Only the beams of the sub's floodlights illuminate the alien environment. Weird, translucent bioluminescent jellyfish, giant tube worms clustered near a hydrothermal vent, black smoker chimney, extreme pressure, abyssal void beyond the light.",
    "A hauntingly sparse photo looking across the deep abyssal plain. Miles of flat, fine grey sediment stretch into the darkness, only illuminated by the external lights of a remote rover. Strange, isolated deep-sea creatures (sea cucumbers, tripod fish) navigate the silent landscape, extreme isolation.",
    "A vibrant, wide-angle underwater photograph of a flourishing coral reef. Divers swim alongside a massive school of colorful reef fish (yellow tangs, clownfish, anthias). Hundreds of diverse coral species (brain coral, staghorn, sea fans) create a kaleidoscope of color and texture under clear, sun-dappled blue water.",
    "A cinematic panoramic view taken from high altitude, looking out over an endless, rolling ocean of fluffy, textured white cumulus clouds. A brilliant, deep blue sky stretches overhead, a solitary commercial airplane flies silently above the cloud layer, sunset casting golden light on the peaks.",
    "A thrilling, dynamic extreme-angle photo taken from directly above a female skydiver. She is in a stable arch position, wearing a colorful suit, falling face-first through a dense, textured bank of fluffy white clouds. The horizon is curved far below, dynamic motion blur on the clouds, bright sunlight.",
    "A low-angle underwater photograph looking up at the colossal dark silhouette of a nuclear attack submarine cruising through deep blue ocean water. A massive screw propeller turns, creating a trail of bubbles. The sun's rays cut down through the water, illuminating the sleek, intimidating metallic hull against the deep blue void.",
    "A powerful, gritty photograph from deep within an operational underground mine shaft. Massive, rough-hewn rock walls, reinforced with steel beams. Heavy-duty yellow mining machinery, dusty air, workers with headlamps creating warm beams of light through the haze, dynamic industrial environment.",
    "A high-resolution orbital photograph capturing a vibrant, Earth-like planet. Swirling white clouds, blue oceans, and green continents are visible. In the background, partially in shadow, a second, larger gas giant planet with concentric rings looms nearby. The Milky Way galaxy serves as the background, sunlight illuminating the scene from the right.",
    "A stunning, cinematic photograph taken from high orbit. A NASA Space Shuttle drifts silently through the black void. Its ceramic heat tiles are entirely covered in vibrant, chaotic graffiti (tags, murals, and symbols) from multiple artists. In the background, the brilliant curved blue and white rim of Earth glows under direct sunlight, contrasting sharply with the defaced machine."
]

In [13]:
for prompt in prompts[15:]:
    pipe.generate(prompt)

  0%|          | 0/35 [00:00<?, ?it/s]

Generated A_vast_high-resolution_satellite_image_2026-05-24T23_58_58.png in 15.22s


  0%|          | 0/35 [00:00<?, ?it/s]

Generated A_cinematic_landscape_shot_from_2026-05-24T23_59_13.png in 15.32s


  0%|          | 0/35 [00:00<?, ?it/s]

...

Generated A_powerful_gritty_photograph_from_2026-05-25T00_04_28.png in 15.84s


  0%|          | 0/35 [00:00<?, ?it/s]

Generated A_high-resolution_orbital_photograph_capturing_2026-05-25T00_04_44.png in 15.74s


  0%|          | 0/35 [00:00<?, ?it/s]

Generated A_stunning_cinematic_photograph_taken_2026-05-25T00_05_00.png in 15.71s
